# Lab 3: Text-to-Cypher Pipeline & GraphRAG
*Building a resilient GraphCypherQAChain with Prompt Engineering.*

---
In our previous labs, we successfully constructed and populated our University Knowledge Graph. However, expecting end-users to write Cypher queries is unrealistic. 

Today, we will build a **GraphRAG (Retrieval-Augmented Generation)** pipeline. We will use an LLM to translate natural language questions into Cypher queries, execute them against Neo4j, and synthesize the results into human-readable answers.

**Objectives:**
1. Connect automatically via credential files (with enhanced schema disabled to simulate a large database).
2. Ingest a baseline University text to guarantee our graph structure.
3. Initialize `GraphCypherQAChain` and identify Text2Cypher failure modes when the LLM cannot see exact property values.
4. Engineer a **Few-Shot Prompt Template** to enforce entity normalization and flexible relationship matching.

In [ ]:
# Install required packages
# %pip install -q -U langchain langchain-neo4j langchain-google-genai google-genai pandas python-dotenv

import os
import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_neo4j import Neo4jGraph, LLMGraphTransformer, GraphCypherQAChain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

print("Packages loaded successfully.")

### 1. Authenticate with Google AI Studio and Neo4j
Ensure you have two text files in the same folder as this Jupyter Notebook:
1. `neo4j_credentials.txt` (Your downloaded AuraDB credentials)
2. `google_credentials.txt` (Create this file and add one line: `GOOGLE_API_KEY=your_api_key_here`)

In [ ]:
print("=== 1. Loading Google AI Studio Credentials ===")

# Load the Google credentials file
load_dotenv("google_credentials.txt")

if os.getenv("GOOGLE_API_KEY"):
    print("SUCCESS! Google API Key found.")
    llm = ChatGoogleGenerativeAI(
        temperature=0, 
        model="gemini-3.6-flash", 
        max_retries=1
    )
else:
    print("\nFAILED: GOOGLE_API_KEY not found. Check your google_credentials.txt file.")

In [ ]:
print("=== 2. Loading Neo4j AuraDB Credentials ===")

# Load the Neo4j credentials file
load_dotenv("neo4j_credentials.txt")

URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USERNAME")
PWD = os.getenv("NEO4J_PASSWORD")

# Automatically patch strict SSL for local Windows environments
if URI and URI.startswith("neo4j+s://"):
    URI = URI.replace("neo4j+s://", "neo4j+ssc://")

try:
    # TURN OFF ENHANCED SCHEMA TO SIMULATE A LARGE ENTERPRISE DB
    # This prevents the LLM from 'cheating' by viewing the exact property strings in our tiny graph
    graph = Neo4jGraph(url=URI, username=USER, password=PWD, enhanced_schema=False)
    graph.refresh_schema()
    print("\nSUCCESS! Connected to Neo4j automatically. Schema refreshed.")
except Exception as e:
    print(f"\nCONNECTION FAILED! Please check your neo4j_credentials.txt file.\nError:\n{e}")

### 2. Baseline Graph Population
To guarantee our QA Chain has data to query, let's instantly parse a short paragraph into Neo4j using the `LLMGraphTransformer` we learned in Lab 6.04.

In [ ]:
print("Clearing existing database and injecting test data...")
graph.query("MATCH (n) DETACH DELETE n")

text = """
Queen's University Belfast focuses on advanced technology education. 
Dr. Alan is a researcher at Queen's University Belfast who specializes in Artificial Intelligence. 
Artificial Intelligence is heavily utilized in GraphRAG systems.
"""
docs = [Document(page_content=text)]

transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["University", "Researcher", "FieldOfStudy", "System"],
    allowed_relationships=["WORKS_AT", "SPECIALIZES_IN", "UTILIZED_IN", "FOCUSES_ON"]
)
graph_docs = transformer.convert_to_graph_documents(docs)
graph.add_graph_documents(graph_docs, baseEntityLabel=True)
graph.refresh_schema()
print("Success! Baseline University graph is ready to query.")

### 3. The Naive Text2Cypher Approach
LangChain provides a default implementation of `GraphCypherQAChain`. Let's test it out-of-the-box using a query with a partial name.

*Note: Because we disabled `enhanced_schema`, the LLM cannot peek at the exact strings in our database. It is flying blind, just like it would in a massive enterprise system. Let's watch it fail to match the partial string 'Queen\'s'.*

In [ ]:
# Initialize the default chain
naive_chain = GraphCypherQAChain.from_llm(
    cypher_llm=llm,
    qa_llm=llm,
    graph=graph,
    verbose=True, 
    allow_dangerous_requests=True
)

# Test 1: A partial string match that fails under naive exact-match Cypher
print("\n--- Query 1: Naive Execution (Partial Match) ---")
response1 = naive_chain.invoke({"query": "who works at Queen's?"})
print("Answer:", response1["result"])

### 4. The Problem: Entity Normalization & Directionality
If a user asks about `Queen's` instead of `Queen's University Belfast`, or if they ask a question that requires traversing an edge backwards, the naive LLM writes an exact-match query (like `{id: "Queen's"}`) that returns empty results. 

To fix this, we engineer a **Few-Shot Prompt Template** that instructs the LLM to:
1. **Normalize Entities:** Use Cypher's `toLower()` function to make case-insensitive comparisons.
2. **Fuzzy Matching:** Use `CONTAINS` instead of strict equality `=` for multi-word concepts.
3. **Flexible Edges:** Use bidirectional relationships `-[]-` if the exact direction is ambiguous in natural language.
4. **Syntax Safety:** Force the LLM to use double quotes (`""`) to prevent Cypher syntax errors with apostrophes.

In [ ]:
cypher_template = """Task: Generate a Cypher statement to query a Neo4j graph database.
Instructions:
1. ONLY use the node labels, properties, and relationship types explicitly provided in the schema.
2. **Entity Normalization:** ALWAYS make string comparisons case-insensitive by using the `toLower()` function. 
   Example: `WHERE toLower(u.id) = toLower("Queen's University Belfast")`
3. **Fuzzy Matching:** If the user provides a partial name, use `CONTAINS`. 
   Example: `WHERE toLower(r.id) CONTAINS toLower("alan")`
4. **Flexible Relationships:** If the direction of the relationship is unclear from the prompt, use bidirectional arrows `-[]-`.
5. Return ONLY the raw Cypher code. Do not wrap it in markdown block quotes (```).
6. **Syntax Safety:** ALWAYS use double quotes ("") for string literals in your Cypher query to prevent errors with apostrophes.

Schema:
{schema}

Few-Shot Examples:
# User: What university does Alan work at?
# Cypher: MATCH (r:Researcher)-[:WORKS_AT]->(u:University) WHERE toLower(r.id) CONTAINS "alan" RETURN u.id

# User: What systems utilize AI?
# Cypher: MATCH (f:FieldOfStudy)-[rel:UTILIZED_IN]-(s:System) WHERE toLower(f.id) CONTAINS "artificial intelligence" RETURN s.id

Question: {question}
Cypher query:"""

# Bind the string to a LangChain PromptTemplate
advanced_prompt = PromptTemplate(
    input_variables=["schema", "question"],
    template=cypher_template
)

print("Advanced Few-Shot Prompt Template configured!")

### 5. Deploying the Resilient GraphRAG Pipeline
Now we inject our `advanced_prompt` into the chain. Watch how the LLM adapts its Cypher generation to handle messy, lowercase user inputs safely.

In [ ]:
# Initialize the chain WITH our custom prompt template
advanced_chain = GraphCypherQAChain.from_llm(
    cypher_llm=llm,
    qa_llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,
    cypher_prompt=advanced_prompt  # <-- INJECTING THE PROMPT HERE
)

# Test: Messy casing and partial names
messy_query = "who works at Queen's?"

print(f"\n--- Processing Messy Query: '{messy_query}' ---")
response2 = advanced_chain.invoke({"query": messy_query})

print("\n*** FINAL ANSWER ***")
print(response2["result"])

---
### 6. Hands-On Student Exercise: System Prompting the QA LLM

By default, the LLM synthesizing the final answer (the `qa_llm`) speaks like a standard, somewhat generic chatbot. However, LangChain allows you to pass a `qa_prompt` to the `GraphCypherQAChain` to strictly control the application's persona and logic rules.

**Your Task:**
Write a system prompt template that forces the QA LLM to answer in the persona of a **Formal University Academic Advisor**. Ensure the prompt explicitly instructs the LLM to treat the provided context as the *absolute truth* from the institutional database, maintaining a highly professional, collegiate tone.

In [ ]:
# 1. Create a custom qa_template string.
my_qa_template = """You are a highly formal University Academic Advisor. You speak in a collegiate, professional, and academic tone. 
The Academic Records provided below represent the exact, verified answer retrieved from the institutional database in response to the student's inquiry. 
Formulate a complete, professional sentence integrating the provided records as the factual answer to the student's question.

Academic Records (Context):
{context}

Student Inquiry: 
{question}

Advisor's Formal Response:"""

# 2. Bind it to a PromptTemplate
custom_qa_prompt = PromptTemplate(
    input_variables=["context", "question"], 
    template=my_qa_template
)

# 3. Re-initialize the chain, passing BOTH your cypher_prompt and qa_prompt
advisor_chain = GraphCypherQAChain.from_llm(
    cypher_llm=llm,
    qa_llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=True,
    cypher_prompt=advanced_prompt, 
    qa_prompt=custom_qa_prompt
)

# 4. Test your new Persona!
print("\n--- Testing the Academic Advisor Persona ---")
response = advisor_chain.invoke({"query": "What systems utilize the field Dr. Alan specializes in?"})
print("\n*** FINAL ANSWER ***")
print(response["result"])